In [1]:
import sys
import os
# Absolute or relative path to the directory you want to import from
module_path = os.path.abspath(os.path.join('..', 'model_exploration'))

if module_path not in sys.path:
    sys.path.append(module_path)


In [3]:
from utils import (
    build_dataset_configs_s1,
    load_and_cache_datasets,
)


from datasets import concatenate_datasets, DatasetDict, Dataset

/app/st-training-workflow/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
configs = build_dataset_configs_s1(None)

In [5]:
configs

{'squad_v2': {'args': {'path': 'rajpurkar/squad_v2'},
  'map_fn': <function utils.build_dataset_configs_s1.<locals>.<lambda>(ex)>,
  'loss': sentence_transformers.losses.MultipleNegativesRankingLoss.MultipleNegativesRankingLoss},
 'wikipedia': {'args': {'path': 'wikimedia/wikipedia',
   'data_dir': '20231101.en'},
  'map_fn': <function utils.build_dataset_configs_s1.<locals>.<lambda>(ex)>,
  'loss': sentence_transformers.losses.MultipleNegativesRankingLoss.MultipleNegativesRankingLoss},
 'StackExchange_Math_titlebody_answer': {'args': {'path': 'flax-sentence-embeddings/stackexchange_math_jsonl',
   'data_dir': 'titlebody_answer'},
  'map_fn': <function utils.build_dataset_configs_s1.<locals>.<lambda>(ex)>,
  'loss': sentence_transformers.losses.MultipleNegativesRankingLoss.MultipleNegativesRankingLoss},
 'StackExchange_Math_title_answer': {'args': {'path': 'flax-sentence-embeddings/stackexchange_math_jsonl',
   'data_dir': 'title_answer'},
  'map_fn': <function utils.build_dataset_conf

In [6]:
ds_dict = load_and_cache_datasets(configs, f"/app/st-training-workflow/data/stage1_cache/NROWS_None", None, 0)

RANK:0;▶ Processing: squad_v2
RANK:0;▶ Processing: wikipedia
RANK:0;▶ Processing: StackExchange_Math_titlebody_answer
RANK:0;▶ Processing: StackExchange_Math_title_answer
RANK:0;▶ Processing: StackExchange_title_body
RANK:0;▶ Processing: StackExchange_Duplicates_titlebody_titlebody
RANK:0;▶ Processing: StackExchange_Duplicates_body_body
RANK:0;▶ Processing: StackExchange_Duplicates_title_title
RANK:0;▶ Processing: Natural_Questions
RANK:0;▶ Processing: PAQ
RANK:0;▶ Processing: Gooaq
RANK:0;▶ Processing: yahoo_title_answers
RANK:0;▶ Processing: msmacro_triplet
RANK:0;▶ Processing: trivia_qa_triplet
RANK:0;▶ Processing: nli_for_simcse_triplet
RANK:0;▶ Processing: quora_dup_triplet
RANK:0;▶ Processing: WikiAnswers
RANK:0;▶ Processing: eli5
RANK:0;▶ Processing: sentence_compression
RANK:0;▶ Processing: Flickr30k_Captions
RANK:0;▶ Processing: Coco_Captions
RANK:0;▶ Processing: xsum
RANK:0;▶ Processing: agnews
RANK:0;▶ Processing: npr
RANK:0;▶ Processing: cnn_dailymail
RANK:0;▶ Processing: c

In [7]:
from tqdm import tqdm

# Create empty datasets for each split
train_pairs = Dataset.from_dict({"anchor": [], "positive": []})
test_pairs = Dataset.from_dict({"anchor": [], "positive": []})
validation_pairs = Dataset.from_dict({"anchor": [], "positive": []})

train_triplets = Dataset.from_dict({"anchor": [], "positive": [], "negative": []})
test_triplets = Dataset.from_dict({"anchor": [], "positive": [], "negative": []})
validation_triplets = Dataset.from_dict({"anchor": [], "positive": [], "negative": []})

for _, ds in tqdm(ds_dict.items()):
    if len(ds["train"].features)== 2:#pairs
        train_pairs = concatenate_datasets([train_pairs, ds["train"]])
        test_pairs = concatenate_datasets([test_pairs, ds["test"]])
        validation_pairs = concatenate_datasets([validation_pairs, ds["validation"]])
    else:
        train_triplets = concatenate_datasets([train_triplets, ds["train"]])
        test_triplets = concatenate_datasets([test_triplets, ds["test"]])
        validation_triplets = concatenate_datasets([validation_triplets, ds["validation"]])


  0%|          | 0/374 [00:00<?, ?it/s]

100%|██████████| 374/374 [02:32<00:00,  2.45it/s]


In [8]:

combined_dataset_pairs = DatasetDict({
    "train": train_pairs,
    "validation": test_pairs,
    "test": validation_pairs
})



combined_dataset_triplets = DatasetDict({
    "train": train_triplets,
    "validation": test_triplets,
    "test": validation_triplets
})


In [9]:
combined_dataset_pairs

DatasetDict({
    train: Dataset({
        features: ['anchor', 'positive'],
        num_rows: 32520187
    })
    validation: Dataset({
        features: ['anchor', 'positive'],
        num_rows: 1806601
    })
    test: Dataset({
        features: ['anchor', 'positive'],
        num_rows: 1806551
    })
})

In [10]:
combined_dataset_triplets

DatasetDict({
    train: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 15664127
    })
    validation: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 890250
    })
    test: Dataset({
        features: ['anchor', 'positive', 'negative'],
        num_rows: 890250
    })
})

In [11]:
combined_dataset_triplets.save_to_disk("../data/stage2_cache/NROWS_None/stage1_triplets")

Saving the dataset (3/3 shards): 100%|██████████| 890250/890250 [00:01<00:00, 822585.21 examples/s] 


In [12]:
combined_dataset_pairs.save_to_disk("../data/stage2_cache/NROWS_None/stage1_pairs")

Saving the dataset (3/3 shards): 100%|██████████| 1806551/1806551 [00:01<00:00, 1118495.80 examples/s]
